# 12 · The suit's senses – OCR and Read

Not everything is a language model. Tony's suit sees, reads and measures. In Foundry, the **same Foundry resource** that hosts your models also offers the **Foundry Tools** (formerly Azure AI Services): purpose-built services for documents, images, speech and more. This chapter reads text out of pictures and PDFs.

| 🔬 Tool (SDK package) | 🎯 Best for | 📥 Input |
|---|---|---|
| **Azure AI Vision – Read** (`azure-ai-vision-imageanalysis`) | fast OCR of photos, signs, whiteboards | images |
| **Document Intelligence – `prebuilt-read`** (`azure-ai-documentintelligence`) | OCR of documents: PDFs, scans, multi-page, handwriting, languages | PDF, images, Office |
| **Document Intelligence – `prebuilt-layout`** | text **plus structure**: tables, headings, selection marks, as Markdown | PDF, images, Office |
| **Content Understanding** (`azure-ai-contentunderstanding`) | LLM-powered extraction with your own schema (chapter 13) | documents, images, audio, video |

All of them use **one endpoint** (`STARK_ACCOUNT_ENDPOINT`), **Entra ID** (no keys) and the *Cognitive Services User* role that `azd up` gave you. Deep dive: [docs/foundry-tools.md](../docs/foundry-tools.md). Sample files come from [scripts/make_sample_docs.py](../scripts/make_sample_docs.py); all content is fictional.

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *

print("Foundry resource endpoint:", cfg("STARK_ACCOUNT_ENDPOINT"))
from IPython.display import Image as Show
Show(filename=str(ROOT / "data" / "docs" / "whiteboard-workshop.png"))

## 1 · Vision Read on a photo
The whiteboard is slightly rotated, like a real photo. Read returns **lines** with a bounding polygon and per-word confidence.

In [ ]:
lines = vision_read("data/docs/whiteboard-workshop.png")
for l in lines:
    print(f"{l['confidence']:.2f}  {l['text']}")

### See what it found
Drawing the polygons is the fastest way to debug OCR.

In [ ]:
from PIL import Image, ImageDraw
img = Image.open(ROOT / "data" / "docs" / "whiteboard-workshop.png").convert("RGB")
d = ImageDraw.Draw(img)
for l in lines:
    d.polygon(l["polygon"], outline="red", width=3)
img

## 2 · Document Intelligence `prebuilt-read` on a PDF
Documents are different from photos: several pages, fonts, languages, maybe handwriting. `AnalyzeDocumentRequest(bytes_source=...)` sends the file; the call is a long-running operation (`begin_...` + `.result()`).

In [ ]:
res = di_analyze("prebuilt-read", "data/docs/invoice-pym-4471.pdf")
print("pages:", len(res.pages), "| languages:", [(l.locale, round(l.confidence, 2)) for l in (res.languages or [])])
print("handwritten spans:", sum(1 for s in (res.styles or []) if s.is_handwritten))
print("---- first lines ----")
for line in res.pages[0].lines[:12]:
    print(line.content)

## 3 · `prebuilt-layout`: text *and* structure
For an invoice the **table** matters. Layout returns tables as cells with row/column indexes, or the whole document as **Markdown** – ideal input for an LLM.

In [ ]:
layout = di_analyze("prebuilt-layout", "data/docs/invoice-pym-4471.pdf", output_content_format="markdown")
print(layout.content[:1200])
print("\ntables found:", len(layout.tables or []))

In [ ]:
import pandas as pd
t = layout.tables[0]
grid = [[""] * t.column_count for _ in range(t.row_count)]
for cell in t.cells:
    grid[cell.row_index][cell.column_index] = cell.content
pd.DataFrame(grid[1:], columns=grid[0])

## 4 · Check the OCR like an engineer
Garbage in, garbage out. For documents you care about, **assert** a few known facts (a mini evaluation – chapter 10 for the big version).

In [ ]:
text_vision = " ".join(l["text"] for l in lines)
expected = ["WORKSHOP", "arc reactor", "Mark 86", "PO-7781", "2026-10-30"]
for e in expected:
    print(("ok   " if e.lower() in text_vision.lower() else "MISS ") + e)

## 🧹 Clean up
Nothing to delete: these calls analyse a file and store nothing in Azure.

## 🎯 Recap
| Situation | Pick |
|---|---|
| A photo or screenshot, just the text | Vision Read |
| PDF/scan, multi-page, languages, handwriting | Document Intelligence `prebuilt-read` |
| You need tables or reading order | `prebuilt-layout` (Markdown) |
| You need *fields* (vendor, total, ...) | prebuilt models (chapter 13) |
| Your own schema, messy or varied documents | Content Understanding (chapter 13) |

⚠️ Documents may contain personal data; Foundry Tools process it in the resource's region. Mind limits (file size, pages per request) and pricing per page.

> 💡 Next: [13 · Invoices and Content Understanding](13_senses_document_intelligence.ipynb).